<a href="https://colab.research.google.com/github/cappellettierica/AprioriMBA/blob/main/MBA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install kaggle pandas numpy matplotlib

In [ ]:
import os
import glob
import zipfile
import pandas as pd
import numpy as np
import time
import math
from math import ceil
from itertools import combinations
from collections import Counter
from typing import Dict, List, Set, Tuple

SUBSTITUTE BACK BEFORE SUBMITTING

In [ ]:
os.environ["KAGGLE_USERNAME"] = "X"
os.environ["KAGGLE_KEY"] = "X"

Download

In [ ]:
!kaggle datasets download -d harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows

Dataset URL: https://www.kaggle.com/datasets/harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows
License(s): CC0-1.0
imdb-dataset-of-top-1000-movies-and-tv-shows.zip: Skipping, found more recently modified local copy (use --force to force download)


Extract files

In [ ]:
with zipfile.ZipFile("imdb-dataset-of-top-1000-movies-and-tv-shows.zip", "r") as zip_ref:
    zip_ref.extractall()

Load the full dataset and verify the columns

In [ ]:
df = pd.read_csv("imdb_top_1000.csv")

print(df.shape)
print(list(df.columns))
df.head()

(1000, 16)
['Poster_Link', 'Series_Title', 'Released_Year', 'Certificate', 'Runtime', 'Genre', 'IMDB_Rating', 'Overview', 'Meta_score', 'Director', 'Star1', 'Star2', 'Star3', 'Star4', 'No_of_Votes', 'Gross']


,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142 min,Drama,9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,"28,341,469"
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175 min,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,"134,966,411"
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152 min,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,"534,858,444"
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202 min,"Crime, Drama",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,"57,300,000"
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96 min,"Crime, Drama",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,"4,360,000"


Control commands

In [159]:
USE_SAMPLE = False
SAMPLE_SIZE = 600 #out of 1000
RANDOM_STATE = 42 #reproducible results

MIN_SUPPORT = 0.002
MAX_K = 4 #maximum itemset size

TOP_N_ITEMSETS = 20

STAR_COLUMNS = ["Star1", "Star2", "Star3", "Star4"]

In [160]:
print("\nNull counts:")
print(df[["Star1", "Star2", "Star3", "Star4"]].isnull().sum())


Null counts:
Star1    0
Star2    0
Star3    0
Star4    0
dtype: int64


Cleaning actors cell

In [161]:
def normalize_actor(value):
    value = str(value).strip()
    return value

Function to transform movies into baskets, cleaning actor cells if needed through the helper function

In [162]:
def row_to_basket(row) -> Set[str]: #from 1 row to a set of strings
    return {
        actor
        for actor in (normalize_actor(row[col]) for col in STAR_COLUMNS)
        if actor is not None
    }

In [163]:
working_df = df.copy() # so i don't edit the original dataset

In [165]:
if USE_SAMPLE:
    working_df = working_df.sample(
        n=min(SAMPLE_SIZE, len(working_df)),
        random_state=RANDOM_STATE
    ).reset_index(drop=True)
else:
    working_df = working_df.reset_index(drop=True)

In [166]:
baskets: List[Set[str]] = []
row_indices_used = []

In [167]:
for idx, row in working_df.iterrows():
    basket = row_to_basket(row) # row to basket
    if basket:
        baskets.append(basket)
        row_indices_used.append(idx)

Inspecting the transformed dataset

In [168]:
num_baskets = len(baskets)
distinct_items = sorted(set().union(*baskets))
num_distinct_items = len(distinct_items)

print(f"Rows: {len(working_df)}")
print(f"Number of baskets: {num_baskets}")
print(f"Distinct actors: {num_distinct_items}")
print("Example baskets:", baskets[:5])

Rows: 1000
Number of baskets: 1000
Distinct actors: 2709
Example baskets: [{'Bob Gunton', 'William Sadler', 'Tim Robbins', 'Morgan Freeman'}, {'Al Pacino', 'James Caan', 'Marlon Brando', 'Diane Keaton'}, {'Aaron Eckhart', 'Heath Ledger', 'Michael Caine', 'Christian Bale'}, {'Al Pacino', 'Diane Keaton', 'Robert Duvall', 'Robert De Niro'}, {'Martin Balsam', 'John Fiedler', 'Lee J. Cobb', 'Henry Fonda'}]


Visual check of the data

In [169]:
display_cols = [c for c in ["Series_Title", "Star1", "Star2", "Star3", "Star4"]]
print(df[display_cols].head(10))

                                    Series_Title              Star1  \
0                       The Shawshank Redemption        Tim Robbins   
1                                  The Godfather      Marlon Brando   
2                                The Dark Knight     Christian Bale   
3                         The Godfather: Part II          Al Pacino   
4                                   12 Angry Men        Henry Fonda   
5  The Lord of the Rings: The Return of the King        Elijah Wood   
6                                   Pulp Fiction      John Travolta   
7                               Schindler's List        Liam Neeson   
8                                      Inception  Leonardo DiCaprio   
9                                     Fight Club          Brad Pitt   

                  Star2              Star3             Star4  
0        Morgan Freeman         Bob Gunton    William Sadler  
1             Al Pacino         James Caan      Diane Keaton  
2          Heath Ledger      

Apriori implementation

In [153]:
Itemset = Tuple[str, ...] # itemset: a tuple of strings

In [170]:
def generate_candidates(prev_frequents: List[Itemset], k: int) -> Set[Itemset]:
    # generate k candidates from k-1 frequent itemsets
    prev_frequents = sorted(prev_frequents)
    prev_set = set(prev_frequents)
    candidates = set()

    n = len(prev_frequents)
    for i in range(n):
        for j in range(i + 1, n):
            a = prev_frequents[i]
            b = prev_frequents[j]

            if a[:k-2] != b[:k-2]:  # join only if the first k-2 items match
              continue # if a pair can't be joined, skip

            candidate = tuple(sorted(set(a) | set(b)))
            if len(candidate) != k:
                continue

            is_valid = True
            # if any (k-1)-subset of a candidate is not frequent, the candidate can't be frequent
            for subset in combinations(candidate, k - 1): # gen all k-1 subsets of the candidate k
                if tuple(sorted(subset)) not in prev_set: # check if frequent subset was freq
                    is_valid = False
                    break

            if is_valid:
                candidates.add(candidate)

    return candidates

Helper function to count singletons

In [171]:
def count_singletons(baskets: List[Set[str]]) -> Counter:
    counts = Counter()
    for basket in baskets:
        counts.update(basket) # adds one count for each actor in the basket
    return counts

Helper function to count how many baskets contain the frequent itemsets

In [172]:
def count_candidates(  # after generating candidate itemsets, count how many baskets contain each one
    baskets: List[Set[str]],
    candidates: Set[Itemset],
    frequent_singletons: Set[str],
    k: int
) -> Counter:
    candidate_set = set(candidates)
    counts = Counter()

    for basket in baskets: # only items that are frequent singletons can participate in frequent larger itemsets
        filtered = sorted(item for item in basket if item in frequent_singletons)
        if len(filtered) < k:
            continue # because if the basket has fewer than k eligible items, it won't contain a k-itemset

        for comb in combinations(filtered, k):
            if comb in candidate_set:
                counts[comb] += 1
        # generate all k-item combinations in the basket, count those that are valid candidates

    return counts

Apriori

In [173]:
def apriori(baskets: List[Set[str]], min_support: float, max_k: int = 4):
    total_baskets = len(baskets)

    min_support_count = ceil(min_support * total_baskets) # convert min support threshold to a required number of baskets
    all_frequents: Dict[int, Dict[Itemset, int]] = {} # dictionary to store freq itemsets by size

    print(f"Total baskets: {total_baskets}")
    print(f"Minimum number of baskets: {min_support_count}")

    # pass 1 -> creates the set of frequent 1-itemsets using the helper function
    singleton_counts = count_singletons(baskets)
    L1 = {
        (item,): count # (item,) stores a singleton as a 1-tuple like ('Actor',)
        for item, count in singleton_counts.items()
        if count >= min_support_count
    }
    L1 = dict(sorted(L1.items(), key=lambda x: (-x[1], x[0]))) # descending and alphabetical count
    all_frequents[1] = L1 # frequent singletons are stored under key 1

    frequent_singletons = set(item[0] for item in L1.keys()) # extracts the actor names from 1 tuples
    prev_frequents = sorted(L1.keys()) # stores the previous level of frequent itemsets

    print(f"Frequent 1-itemsets: {len(L1)}")

    # Passes 2 to 4
    for k in range(2, max_k + 1):
        if not prev_frequents: # if the previous level has 0 frequent itemsets, no larger levels
            all_frequents[k] = {}
            print(f"Frequent {k}-itemsets: 0")
            continue # if no candidates, no frequent itemsets at this level or above
        # potential frequent itemsets of size k
        candidates = generate_candidates(prev_frequents, k)
        print(f"Candidate {k}-itemsets: {len(candidates)}")

        if not candidates:
            all_frequents[k] = {}
            print(f"Frequent {k}-itemsets: 0")
            prev_frequents = []
            continue

        candidate_counts = count_candidates(
            baskets=baskets,
            candidates=candidates,
            frequent_singletons=frequent_singletons,
            k=k
        ) # scan the baskets, count candidate supports

        Lk = { # filter candidates down to truly frequent itemsets
            itemset: count
            for itemset, count in candidate_counts.items()
            if count >= min_support_count
        }
        Lk = dict(sorted(Lk.items(), key=lambda x: (-x[1], x[0])))
        all_frequents[k] = Lk
        prev_frequents = sorted(Lk.keys())

        print(f"Frequent {k}-itemsets: {len(Lk)}")
        # store current results before going into next pass

    return all_frequents, min_support_count

Run apriori

In [174]:
start = time.time()

frequent_itemsets, min_support_count = apriori(
    baskets=baskets,
    min_support=MIN_SUPPORT,
    max_k=MAX_K
)

end = time.time()
total_time = end - start
print(f"Ttotal run time: {total_time} seconds")

Total baskets: 1000
Minimum number of baskets: 2
Frequent 1-itemsets: 641
Candidate 2-itemsets: 205120
Frequent 2-itemsets: 121
Candidate 3-itemsets: 26
Frequent 3-itemsets: 25
Candidate 4-itemsets: 4
Frequent 4-itemsets: 4
Ttotal run time: 0.5438437461853027 seconds


Show the frequent itemsets

In [ ]:
def itemsets_to_dataframe(
    frequent_itemsets: Dict[int, Dict[Itemset, int]],
    total_baskets: int
) -> pd.DataFrame:
    rows = []
    for k, itemsets_k in frequent_itemsets.items():
        for itemset, count in itemsets_k.items():
            rows.append({
                "k": k,
                "itemset": ", ".join(itemset),
                "support_count": count,
                "support": count / total_baskets
            })

    result = pd.DataFrame(rows)
    if not result.empty:
        result = result.sort_values(
            ["k", "support_count", "itemset"],
            ascending=[True, False, True]
        ).reset_index(drop=True)
    return result

for k in sorted(frequent_itemsets.keys()):
    print(f"\nTop frequent {k}-itemsets")
    top_rows = list(frequent_itemsets[k].items())[:TOP_N_ITEMSETS]
    if not top_rows:
        print("  None")
        continue
    for itemset, count in top_rows:
        print(f"  {itemset} -> count={count}, support={count / num_baskets:.4f}")


Top frequent 1-itemsets
  ('Chris Evans',) -> count=3, support=0.0300
  ('Kirk Douglas',) -> count=3, support=0.0300
  ('Ben Affleck',) -> count=2, support=0.0200
  ('Benicio Del Toro',) -> count=2, support=0.0200
  ('Brad Pitt',) -> count=2, support=0.0200
  ('Bryan Cranston',) -> count=2, support=0.0200
  ('F. Murray Abraham',) -> count=2, support=0.0200
  ('Farhan Akhtar',) -> count=2, support=0.0200
  ('Jake Gyllenhaal',) -> count=2, support=0.0200
  ('Joaquin Phoenix',) -> count=2, support=0.0200
  ('Joe Russo',) -> count=2, support=0.0200
  ('John Goodman',) -> count=2, support=0.0200
  ('Jose Coronado',) -> count=2, support=0.0200
  ('Laurence Olivier',) -> count=2, support=0.0200
  ('Leonardo DiCaprio',) -> count=2, support=0.0200
  ('Mark Ruffalo',) -> count=2, support=0.0200
  ('Michael Caine',) -> count=2, support=0.0200
  ('Rachel McAdams',) -> count=2, support=0.0200
  ('Ralph Fiennes',) -> count=2, support=0.0200
  ('Robert De Niro',) -> count=2, support=0.0200

Top freq